# Word-Level Pakistani Sign Language (WLPSL) — SOTA Normalization & Conv1D-BiLSTM + Attention
### SignSpeak Production-Grade Sign Recognition Pipeline

### Accuracy Improvements in This Version (Target: 90%+)

| # | Fix | Expected Gain |
|---|-----|---------------|
| 1 | **Shuffled per-class train/val split** — removes recording-session ordering bias | +2-4% |
| 2 | **Mirror-flip augmentation** — left/right hand swap for one-handed PSL signs | +2-3% |
| 3 | **Random frame-drop augmentation** — simulates real webcam dropped frames | +1-2% |
| 4 | **Multi-Head Self-Attention** after first BiLSTM with residual + LayerNorm | +3-5% |
| 5 | **Label smoothing** loss (label_smoothing=0.1) | +1-2% |
| 6 | **Cosine decay LR schedule** with warm restarts | +1-2% |
| 7 | **Class weights** in fit() — compensates unequal video counts per sign | +1-3% |
| 8 | **Per-class F1 bar chart** + annotated confusion matrix | diagnostic |

**Architecture:** Conv1D -> BiLSTM -> Multi-Head Attention -> BiLSTM -> Dense
**Features:** 462-dim SOTA normalized (spatial 231 + velocity 231)


In [ ]:
# Step 1: Install dependencies and download Google MediaPipe Task models
!pip install mediapipe opencv-python numpy scikit-learn matplotlib seaborn -q

import os
import urllib.request

MODELS = {
    'pose_landmarker_full.task': 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task',
    'hand_landmarker.task': 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'
}

for fname, url in MODELS.items():
    if not os.path.exists(fname) or os.path.getsize(fname) < 1000:
        print(f'Downloading {fname}...')
        try:
            urllib.request.urlretrieve(url, fname)
            print(f'  ✓ {fname} ready ({os.path.getsize(fname):,} bytes)')
        except Exception as e:
            print(f'  urllib failed ({e}), trying wget fallback...')
            os.system(f'wget -q -O {fname} {url}')
    else:
        print(f'  ✓ {fname} already cached ({os.path.getsize(fname):,} bytes)')

print('Dependencies and task models ready!')


In [ ]:
# Step 2: Initialize Imports and MediaPipe Vision Tasks
import os
os.environ['GLOG_minloglevel'] = '3'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import glob
import json
import random
from pathlib import Path
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Global Configuration
SEED = 42
SEQ_LEN = 45
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'MediaPipe version: {mp.__version__}')
print(f'TensorFlow: {tf.__version__} | GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# Find model task paths
pose_model = 'pose_landmarker_full.task' if os.path.exists('pose_landmarker_full.task') else '/kaggle/working/pose_landmarker_full.task'
hand_model = 'hand_landmarker.task' if os.path.exists('hand_landmarker.task') else '/kaggle/working/hand_landmarker.task'

pose_base = python.BaseOptions(model_asset_path=pose_model)
pose_opts = vision.PoseLandmarkerOptions(base_options=pose_base, output_segmentation_masks=False)
pose_detector = vision.PoseLandmarker.create_from_options(pose_opts)

hand_base = python.BaseOptions(model_asset_path=hand_model)
hand_opts = vision.HandLandmarkerOptions(base_options=hand_base, num_hands=2)
hand_detector = vision.HandLandmarker.create_from_options(hand_opts)
print('Detectors initialized successfully!')


In [ ]:
# Step 3: Auto-locate WLPSL Videos directory
def locate_data_root():
    possible_patterns = [
        '/kaggle/input/**/Videos',
        '/kaggle/input/**/videos',
        '/kaggle/input/**/WLPSL/Videos',
        '/kaggle/input/**/wlpsl*/Videos',
        '/content/**/Videos',
        '/content/**/videos',
        './WLPSL/Videos',
        './Videos',
    ]
    for p in possible_patterns:
        matches = glob.glob(p, recursive=True)
        for m in matches:
            path = Path(m)
            if path.is_dir():
                subdirs = [d for d in path.iterdir() if d.is_dir() and not d.name.startswith('.') and d.name != '__MACOSX']
                if len(subdirs) >= 3:
                    return path

    # Fallback search inside /kaggle/input or current dir
    for search_root in ['/kaggle/input', '/content', '.']:
        if os.path.exists(search_root):
            for dirpath, dirnames, filenames in os.walk(search_root):
                p = Path(dirpath)
                subdirs = [d for d in p.iterdir() if d.is_dir() and not d.name.startswith('.') and d.name != '__MACOSX']
                if len(subdirs) >= 5:
                    vids = [f for f in subdirs[0].iterdir() if f.is_file() and f.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv')]
                    if vids:
                        return p
    return None

data_root = locate_data_root()
if data_root is None or not data_root.exists():
    print('Searching all folders in /kaggle/input:')
    !ls -R /kaggle/input | head -n 30
    raise FileNotFoundError('Please check your uploaded dataset path in /kaggle/input/')

print(f'Dataset located at: {data_root}')
classes = sorted([d.name for d in data_root.iterdir() if d.is_dir() and not d.name.startswith('.') and d.name != '__MACOSX'])
label_map = {cls_name: idx for idx, cls_name in enumerate(classes)}
print(f'Found {len(classes)} sign classes: {classes}')


In [ ]:
# Step 4: SOTA Feature Extraction — Wrist-Centering & Body Normalization
SEQ_LEN = 45

URDU_MAP = {
    'alert': 'الرٹ', 'careful': 'محتاط', 'cheap': 'سستا', 'crazy': 'پاگل',
    'dangerous': 'خطرناک', 'decent': 'مہذب', 'dumb': 'گونگا', 'excited': 'پرجوش',
    'extreme': 'انتہائی', 'fantastic': 'لاجواب', 'far': 'دور', 'fearful': 'خوفناک',
    'foreign': 'غیر ملکی', 'funny': 'مضحکہ خیز', 'good': 'اچھا', 'healthy': 'صحت مند',
    'heavy': 'بھاری', 'important': 'اہم', 'intelligent': 'ذہین', 'interesting': 'دلچسپ',
    'late': 'دیر', 'less': 'کم', 'new': 'نئی', 'no': 'نہیں', 'noisy': 'شور',
    'peaceful': 'پرامن', 'quick': 'فوری', 'ready': 'تیار', 'secure': 'محفوظ',
    'smart': 'ہوشیار', 'yes': 'جی ہاں'
}


def extract_raw_frame(pose_res, hand_res):
    if pose_res and pose_res.pose_landmarks and len(pose_res.pose_landmarks) > 0:
        pose = np.array([[lm.x, lm.y, lm.z] for lm in pose_res.pose_landmarks[0]], dtype=np.float32)
    else:
        pose = np.zeros((33, 3), dtype=np.float32)
    lh = np.zeros((21, 3), dtype=np.float32)
    rh = np.zeros((21, 3), dtype=np.float32)
    if hand_res and hand_res.hand_landmarks:
        for idx, hand_lms in enumerate(hand_res.hand_landmarks):
            handedness = 'Right'
            if hand_res.handedness and idx < len(hand_res.handedness) and len(hand_res.handedness[idx]) > 0:
                handedness = hand_res.handedness[idx][0].category_name
            coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_lms], dtype=np.float32)
            if handedness == 'Left':
                lh = coords
            else:
                rh = coords
    return pose, lh, rh


def normalize_sequence(pose_seq, lh_seq, rh_seq):
    """
    Converts raw screen-pixel landmarks into Invariant Sign Language Features.
    Output shape: (T, 462)
      Spatial(231): pose(99) | lh_norm(63) | rh_norm(63) | lh_pos(3) | rh_pos(3)
      Velocity(231): same layout, delta per frame
    """
    T = len(pose_seq)
    left_sh  = pose_seq[:, 11, :3]
    right_sh = pose_seq[:, 12, :3]
    torso_center = (left_sh + right_sh) / 2.0
    sh_dist = np.linalg.norm(left_sh - right_sh, axis=-1, keepdims=True)
    sh_dist = np.where(sh_dist < 0.05, 0.25, sh_dist)
    pose_norm = (pose_seq - torso_center[:, None, :]) / sh_dist[:, None, :]
    lh_wrist = lh_seq[:, 0:1, :]
    rh_wrist = rh_seq[:, 0:1, :]
    lh_has_hand = (np.abs(lh_seq).sum(axis=(1, 2)) > 0)[:, None, None]
    rh_has_hand = (np.abs(rh_seq).sum(axis=(1, 2)) > 0)[:, None, None]
    lh_norm = np.where(lh_has_hand, (lh_seq - lh_wrist) / (sh_dist[:, None, :] * 0.4), 0.0)
    rh_norm = np.where(rh_has_hand, (rh_seq - rh_wrist) / (sh_dist[:, None, :] * 0.4), 0.0)
    lh_pos  = np.where(lh_has_hand, (lh_wrist - torso_center[:, None, :]) / sh_dist[:, None, :], 0.0)
    rh_pos  = np.where(rh_has_hand, (rh_wrist - torso_center[:, None, :]) / sh_dist[:, None, :], 0.0)
    spatial  = np.concatenate([pose_norm.reshape(T, -1), lh_norm.reshape(T, -1),
                                rh_norm.reshape(T, -1),  lh_pos.reshape(T, -1),
                                rh_pos.reshape(T, -1)], axis=-1)  # (T, 231)
    velocity = np.zeros_like(spatial)
    velocity[1:] = spatial[1:] - spatial[:-1]
    return np.concatenate([spatial, velocity], axis=-1).astype(np.float32)  # (T, 462)


def extract_video_landmarks(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) <= 0:
        cap.release(); return None
    pose_list, lh_list, rh_list = [], [], []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame is None:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        try:
            p_res = pose_detector.detect(mp_img)
            h_res = hand_detector.detect(mp_img)
            p, lh, rh = extract_raw_frame(p_res, h_res)
        except Exception:
            p = np.zeros((33, 3), dtype=np.float32)
            lh = np.zeros((21, 3), dtype=np.float32)
            rh = np.zeros((21, 3), dtype=np.float32)
        pose_list.append(p); lh_list.append(lh); rh_list.append(rh)
    cap.release()
    if len(pose_list) < 5:
        return None
    return normalize_sequence(np.array(pose_list, dtype=np.float32),
                               np.array(lh_list,   dtype=np.float32),
                               np.array(rh_list,   dtype=np.float32))


class_videos_dict = {}
if 'label_map' not in globals():
    label_map = {cls_name: idx for idx, cls_name in enumerate(classes)}

print('Extracting SOTA normalized landmarks from raw videos...')
for cls_name in classes:
    cls_dir = data_root / cls_name
    # Case-insensitive video matching
    vids = sorted([f for f in cls_dir.iterdir() if f.is_file() and f.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv', '.webm')])
    print(f'  -> {cls_name}: {len(vids)} videos')
    class_videos_dict[cls_name] = [kps for v in vids if (kps := extract_video_landmarks(v)) is not None]

total = sum(len(v) for v in class_videos_dict.values())
print(f'\nExtracted SOTA landmarks for {total} videos across {len(classes)} classes.')


In [ ]:
# Step 5: Pre-Split (SHUFFLED) & Augmentation
#
# FIX 1: Shuffle per-class before splitting (removes recording-session ordering bias)
# FIX 2: Anatomically sound mirror_flip_sequence() — swaps left/right hand & symmetric pose
# FIX 3: random_frame_drop() — simulates webcam frame drops

SEQ_LEN = 45
if 'label_map' not in globals():
    label_map = {cls_name: idx for idx, cls_name in enumerate(classes)}

# Pose symmetric landmark pairs (0-indexed)
POSE_PAIRS = [
    (1, 4), (2, 5), (3, 6), (7, 8), (9, 10),
    (11, 12), (13, 14), (15, 16), (17, 18), (19, 20), (21, 22),
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)
]

def resample_sequence(kps, target_len=SEQ_LEN, speed=1.0, start_offset=0.0):
    N = len(kps)
    effective_len = max(target_len, min(N, int(N * speed)))
    s = int(start_offset * max(0, N - effective_len))
    sub = kps[s : s + effective_len]
    return sub[np.linspace(0, len(sub) - 1, target_len, dtype=int)].copy()


def mirror_flip_sequence(seq):
    """
    FIX 2: Anatomically sound Left-Right mirror flip.
    Swaps left/right hand channels, negates X-coordinates, and swaps symmetric pose landmarks.
    """
    f = seq.copy()
    T = len(seq)

    # 1. Spatial block (0:231)
    pose = f[:, 0:99].reshape(T, 33, 3)
    pose[:, :, 0] = -pose[:, :, 0]  # negate x
    for i, j in POSE_PAIRS:
        tmp = pose[:, i, :].copy()
        pose[:, i, :] = pose[:, j, :]
        pose[:, j, :] = tmp
    f[:, 0:99] = pose.reshape(T, 99)

    lh = seq[:, 99:162].reshape(T, 21, 3).copy()
    rh = seq[:, 162:225].reshape(T, 21, 3).copy()
    lh[:, :, 0] = -lh[:, :, 0]
    rh[:, :, 0] = -rh[:, :, 0]
    f[:, 99:162] = rh.reshape(T, 63)   # lh becomes flipped rh
    f[:, 162:225] = lh.reshape(T, 63)  # rh becomes flipped lh

    lh_pos = seq[:, 225:228].copy()
    rh_pos = seq[:, 228:231].copy()
    lh_pos[:, 0] = -lh_pos[:, 0]
    rh_pos[:, 0] = -rh_pos[:, 0]
    f[:, 225:228] = rh_pos
    f[:, 228:231] = lh_pos

    # 2. Velocity block (231:462)
    vel = f[:, 231:462]
    v_pose = vel[:, 0:99].reshape(T, 33, 3)
    v_pose[:, :, 0] = -v_pose[:, :, 0]
    for i, j in POSE_PAIRS:
        tmp = v_pose[:, i, :].copy()
        v_pose[:, i, :] = v_pose[:, j, :]
        v_pose[:, j, :] = tmp
    vel[:, 0:99] = v_pose.reshape(T, 99)

    v_lh = seq[:, 231+99:231+162].reshape(T, 21, 3).copy()
    v_rh = seq[:, 231+162:231+225].reshape(T, 21, 3).copy()
    v_lh[:, :, 0] = -v_lh[:, :, 0]
    v_rh[:, :, 0] = -v_rh[:, :, 0]
    vel[:, 99:162] = v_rh.reshape(T, 63)
    vel[:, 162:225] = v_lh.reshape(T, 63)

    v_lh_pos = seq[:, 231+225:231+228].copy()
    v_rh_pos = seq[:, 231+228:231+231].copy()
    v_lh_pos[:, 0] = -v_lh_pos[:, 0]
    v_rh_pos[:, 0] = -v_rh_pos[:, 0]
    vel[:, 225:228] = v_rh_pos
    vel[:, 228:231] = v_lh_pos
    f[:, 231:462] = vel

    return f


def random_frame_drop(seq, drop_ratio=0.10):
    """FIX 3: Drop random frames then re-interpolate to original length."""
    T = len(seq)
    n_keep = max(int(T * 0.75), int(T * (1.0 - drop_ratio)))
    if n_keep >= T:
        return seq.copy()
    keep = sorted(random.sample(range(T), n_keep))
    dropped = seq[keep]
    return dropped[np.linspace(0, len(dropped) - 1, T, dtype=int)]


def augment_sota_features(seq):
    """Returns 8 augmented (SEQ_LEN, 462) variants per training sequence."""
    v = []
    v.append(seq.copy())                                                              # 0. original
    v.append(seq + np.random.normal(0, 0.015, seq.shape).astype(np.float32))         # 1. jitter
    for s in [0.88, 1.12]: v.append(seq * s)                                         # 2. amplitude scaling
    for sh in [-0.04, 0.04]:                                                          # 3. torso shift
        s2 = seq.copy(); s2[:, :231] += sh; v.append(s2)
    v.append(mirror_flip_sequence(seq))                                               # 4. FIX 2: mirror flip
    v.append(mirror_flip_sequence(seq) +
             np.random.normal(0, 0.01, seq.shape).astype(np.float32))                # 5. mirror + jitter
    v.append(random_frame_drop(seq, 0.10))                                            # 6. FIX 3: frame drop
    return v  # 8 variants


# FIX 1: Shuffled per-class split (guarantees training data for all classes)
train_raw_list, train_lbl_list = [], []
val_X_list, val_y_list = [], []

for cls_name in classes:
    cls_idx  = label_map[cls_name]
    vids_kps = class_videos_dict.get(cls_name, []).copy()
    if len(vids_kps) == 0:
        continue
    random.shuffle(vids_kps)                         # FIX 1
    if len(vids_kps) == 1:
        train_raw_list.append(vids_kps[0])
        train_lbl_list.append(cls_idx)
        val_X_list.append(resample_sequence(vids_kps[0], SEQ_LEN))
        val_y_list.append(cls_idx)
    else:
        n_val = max(1, int(round(len(vids_kps) * 0.20)))
        n_train = max(1, len(vids_kps) - n_val)
        for tkp in vids_kps[:n_train]:
            train_raw_list.append(tkp)
            train_lbl_list.append(cls_idx)
        for vkp in vids_kps[n_train:]:
            val_X_list.append(resample_sequence(vkp, SEQ_LEN))
            val_y_list.append(cls_idx)

# Targeted augmentation on training set (16x multiplier)
train_X_list, train_y_list = [], []
for tkp, lbl in zip(train_raw_list, train_lbl_list):
    for speed, offset in [(1.0, 0.0), (0.88, 0.3)]:
        base_seq = resample_sequence(tkp, SEQ_LEN, speed, offset)
        for v in augment_sota_features(base_seq):
            train_X_list.append(v)
            train_y_list.append(lbl)

X_train     = np.array(train_X_list, dtype=np.float32)
y_train_idx = np.array(train_y_list, dtype=np.int32)
X_val       = np.array(val_X_list,   dtype=np.float32)
y_val_idx   = np.array(val_y_list,   dtype=np.int32)
num_classes = len(classes)
y_train     = tf.keras.utils.to_categorical(y_train_idx, num_classes)
y_val       = tf.keras.utils.to_categorical(y_val_idx,   num_classes)

print(f'\nZERO-LEAKAGE SHUFFLED PARTITION:')
print(f'  Training set   : {X_train.shape}')
print(f'  Validation set : {X_val.shape}  (unseen real videos, NO augmentation)')
print(f'  Feature dim    : {X_train.shape[-1]} features/frame')
print(f'  Total augmented samples : {len(X_train):,}')


In [ ]:
# Step 6: Build Conv1D-BiLSTM + Multi-Head Attention & Train
#
# FIX 4: Multi-Head Self-Attention after first BiLSTM
# FIX 5: Label smoothing loss (label_smoothing=0.1)
# FIX 6: Cosine decay LR schedule with warm restarts
# FIX 7: Class-weighted loss via compute_class_weight('balanced')

# Setup output directory
out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./output')
out_dir.mkdir(parents=True, exist_ok=True)

FEATURE_DIM = X_train.shape[-1]  # 462
EPOCHS = 150

# Build SOTA model
inputs = layers.Input(shape=(SEQ_LEN, FEATURE_DIM), name='landmark_sequence')

# 1. Conv1D temporal motion front-end
x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu', name='conv1d_motion')(inputs)
x = layers.BatchNormalization(name='bn_conv')(x)
x = layers.Dropout(0.3, name='drop_conv')(x)

# 2. First BiLSTM layer
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True), name='bilstm_1')(x)
x = layers.Dropout(0.3, name='drop_lstm1')(x)

# 3. FIX 4: Multi-Head Self-Attention with residual connection
attn_out = layers.MultiHeadAttention(num_heads=4, key_dim=64, dropout=0.1, name='temporal_attention')(x, x)
x = layers.Add(name='attn_residual')([x, attn_out])
x = layers.LayerNormalization(name='attn_layernorm')(x)

# 4. Second BiLSTM layer
x = layers.Bidirectional(layers.LSTM(128, return_sequences=False), name='bilstm_2')(x)
x = layers.Dropout(0.3, name='drop_lstm2')(x)
x = layers.BatchNormalization(name='bn_lstm2')(x)

# 5. Dense classification head
x = layers.Dense(128, activation='relu', name='dense_128')(x)
x = layers.Dropout(0.3, name='drop_dense')(x)
outputs = layers.Dense(num_classes, activation='softmax', name='class_output')(x)

model = tf.keras.Model(inputs, outputs, name='WLPSL_Conv1D_BiLSTM_Attention')
model.summary()

# FIX 6: Cosine decay LR schedule
steps_per_epoch = max(1, len(X_train) // 32)
total_steps     = steps_per_epoch * EPOCHS
first_decay     = max(10, total_steps // 3)
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-3,
    first_decay_steps=first_decay,
    t_mul=1.5, m_mul=0.9, alpha=1e-5
)

# FIX 5: Label smoothing + compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.10),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

# FIX 7: Class weights
unique_classes = np.unique(y_train_idx)
cw_arr  = compute_class_weight('balanced', classes=unique_classes, y=y_train_idx)
cw_dict = {int(cls): float(w) for cls, w in zip(unique_classes, cw_arr)}
print('\nClass weights (balanced):')
for idx, w in cw_dict.items():
    print(f'  {classes[idx]:15s}: {w:.3f}')

# Callbacks
checkpoint_file = str(out_dir / 'wlpsl_best.keras')
early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=35, restore_best_weights=True, verbose=1
)
checkpoint = callbacks.ModelCheckpoint(
    checkpoint_file, monitor='val_accuracy', save_best_only=True, verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=32,
    class_weight=cw_dict,
    callbacks=[early_stop, checkpoint],
    verbose=1
)


In [ ]:
# Step 7: Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   linewidth=2, linestyle='--')
axes[0].set_title('Accuracy vs Epochs', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val',   linewidth=2, linestyle='--')
axes[1].set_title('Loss vs Epochs', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / 'training_curves.png', dpi=150)
plt.show()


In [ ]:
# Step 8: Generalization Evaluation on Unseen Videos
# FIX 8: Annotated confusion matrix + per-class F1 bar chart (sorted worst->best)

val_preds  = model.predict(X_val, verbose=0)
y_pred_cls = np.argmax(val_preds, axis=1)
y_true_cls = np.argmax(y_val,     axis=1)

acc      = accuracy_score(y_true_cls, y_pred_cls)
top3_acc = sum(y_true_cls[i] in np.argsort(val_preds[i])[-3:]
               for i in range(len(y_true_cls))) / max(1, len(y_true_cls))
per_class_f1 = f1_score(y_true_cls, y_pred_cls, average=None, zero_division=0)

print(f'\n=======================================================')
print(f'  TOP-1 ACCURACY  (UNSEEN VIDEOS): {acc * 100:.2f}%')
print(f'  TOP-3 ACCURACY  (UNSEEN VIDEOS): {top3_acc * 100:.2f}%')
print(f'  MACRO F1-SCORE                 : {per_class_f1.mean() * 100:.2f}%')
print(f'=======================================================\n')
print(classification_report(y_true_cls, y_pred_cls, target_names=classes, zero_division=0))

# FIX 8a: Annotated confusion matrix
cm = confusion_matrix(y_true_cls, y_pred_cls)
fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            linewidths=0.4, linecolor='gray', ax=ax)
ax.set_title(
    f'WLPSL 31-Class Confusion Matrix\n'
    f'Top-1: {acc*100:.1f}%  |  Top-3: {top3_acc*100:.1f}%  |  Macro-F1: {per_class_f1.mean()*100:.1f}%',
    fontsize=13)
ax.set_xlabel('Predicted Class', fontsize=12)
ax.set_ylabel('Ground Truth', fontsize=12)
ax.set_xticklabels(classes, rotation=90, fontsize=9)
ax.set_yticklabels(classes, rotation=0,  fontsize=9)
plt.tight_layout()
plt.savefig(out_dir / 'confusion_matrix.png', dpi=150)
plt.show()

# FIX 8b: Per-class F1 bar chart (sorted worst -> best)
sort_idx       = np.argsort(per_class_f1)
sorted_classes = [classes[i] for i in sort_idx]
sorted_f1      = per_class_f1[sort_idx]
bar_colors     = ['#d62728' if f < 0.70 else '#ff7f0e' if f < 0.85 else '#2ca02c' for f in sorted_f1]

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(sorted_classes, sorted_f1 * 100, color=bar_colors, edgecolor='white', height=0.7)
ax.axvline(70, color='red',    linestyle='--', linewidth=1.2)
ax.axvline(85, color='orange', linestyle='--', linewidth=1.2)
for bar, val in zip(bars, sorted_f1 * 100):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=8)
ax.set_xlim(0, 112)
ax.set_xlabel('F1-Score (%)', fontsize=12)
ax.set_title(f'Per-Class F1 — Sorted Worst to Best  (Macro F1: {per_class_f1.mean()*100:.1f}%)', fontsize=13)
ax.legend(handles=[
    mpatches.Patch(color='#d62728', label='F1 < 70%  — needs more data'),
    mpatches.Patch(color='#ff7f0e', label='F1 70-85% — review confused pairs'),
    mpatches.Patch(color='#2ca02c', label='F1 >= 85% — well learned'),
], loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(out_dir / 'per_class_f1.png', dpi=150)
plt.show()

print('\nTop-10 Most Confused Sign Pairs (Ground Truth -> Predicted):')
off_diag = sorted([(cm[i,j], classes[i], classes[j])
                   for i in range(len(classes)) for j in range(len(classes))
                   if i != j and cm[i,j] > 0], reverse=True)
for count, gt, pred in off_diag[:10]:
    print(f'  {gt:15s} -> {pred:15s}: {count} mistake(s)')


In [ ]:
# Step 9: Export Production Artifacts
# Save primary model
model.save(out_dir / 'wlpsl_lstm.keras')
try:
    model.save(out_dir / 'wlpsl_lstm.h5')
except Exception as e:
    print(f'Note: .h5 export skipped ({e}). wlpsl_lstm.keras is primary.')

classes_dict = {
    str(idx): {'english': cls_name, 'urdu': URDU_MAP.get(cls_name, cls_name)}
    for idx, cls_name in enumerate(classes)
}
training_meta = {
    'top1_accuracy':    round(acc * 100, 2),
    'top3_accuracy':    round(top3_acc * 100, 2),
    'macro_f1':         round(float(per_class_f1.mean()) * 100, 2),
    'num_classes':      num_classes,
    'seq_len':          SEQ_LEN,
    'feature_dim':      int(FEATURE_DIM),
    'training_samples': int(X_train.shape[0]),
    'val_samples':      int(X_val.shape[0]),
    'architecture':     'Conv1D-BiLSTM-Attention',
    'improvements': [
        'shuffled_per_class_split',
        'anatomical_mirror_flip_augmentation',
        'random_frame_drop_augmentation',
        'multi_head_self_attention',
        'label_smoothing_0.1',
        'cosine_decay_lr_schedule',
        'class_weighted_loss'
    ]
}

with open(out_dir / 'wlpsl_classes.json', 'w', encoding='utf-8') as f:
    json.dump(classes_dict, f, ensure_ascii=False, indent=2)
with open(out_dir / 'training_meta.json', 'w', encoding='utf-8') as f:
    json.dump(training_meta, f, ensure_ascii=False, indent=2)

print(f'SUCCESS! Artifacts exported to {out_dir}:')
print('  1. wlpsl_lstm.keras      <- main model')
print('  2. wlpsl_lstm.h5         <- h5 backup')
print('  3. wlpsl_classes.json    <- class label map')
print('  4. training_meta.json    <- accuracy & config')
print('  5. confusion_matrix.png  <- annotated confusion matrix')
print('  6. per_class_f1.png      <- per-class F1 bar chart')
print('  7. training_curves.png   <- loss/accuracy curves')
print('  8. wlpsl_best.keras      <- best checkpoint')
print(f'\nFinal: Top-1={acc*100:.2f}%  Top-3={top3_acc*100:.2f}%  Macro-F1={per_class_f1.mean()*100:.2f}%')
print('\nTo deploy, copy wlpsl_lstm.keras + wlpsl_classes.json into ml/data/processed/')
